# Task1 — 질문유형 분류기 (데이터 생성 → 학습 → 평가 → 예측)

라벨: `졸업요건=0, 학교공지=1, 학사일정=2, 식단=3, 통학/셔틀=4`

이 노트북은 **위에서 아래로 한 번 실행하면 분류기를 처음부터 만들어낸다.**

1. 학습 데이터 생성 — `scripts/build_cls_dataset.py` 가 템플릿 x 슬롯으로 train/valid 생성
2. 학습 — `klue/roberta-base` 파인튜닝 후 `model/` 에 저장
3. 평가 — `valid`(템플릿) + `eval_natural`(사람이 쓴 구어체)로 일반화까지 확인
4. 예측 — `data/test_cls.json` → `outputs/cls_output.json`

> 분류기 가중치는 용량 때문에 `.gitignore` 로 제외돼 있다(`model/*.safetensors`).
> 그래서 clone 직후에는 가중치가 없고, **이 노트북이 직접 학습해서 만든다.**
> `chatbot.sh` 도 같은 `model/` 을 쓰므로, 이 노트북을 먼저 돌려두면 챗봇이 바로 그 분류기를 쓴다.
> (순서를 잊어도 `chatbot.sh` 가 가중치 없음을 감지해 스스로 학습한다.)

In [ ]:
# [setup] 의존성 — 과제 사양 버전(torch 2.5.1 / torchvision 0.20.1 / pytorch-lightning 2.4.0).
# 이미 맞으면 건너뛴다. torch 를 import 하지 않고 메타데이터 버전만 확인한다.
import importlib.metadata as _md
import subprocess
import sys
from pathlib import Path


def _ver(pkg):
    try:
        return _md.version(pkg)
    except Exception:
        return None


_req = Path('requirements.txt') if Path('requirements.txt').exists() else Path('../requirements.txt')
if _ver('torch') != '2.5.1' or _ver('transformers') is None:
    print('[setup] torch', _ver('torch'), '/ transformers', _ver('transformers'), '→ requirements 설치. 잠시 걸립니다...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(_req)])
    print('[setup] 설치 완료. 버전이 여전히 안 맞으면 런타임 > 세션 다시 시작 후 이 셀부터 다시 실행하세요.')
else:
    print('[setup] torch 2.5.1 / transformers', _ver('transformers'), '확인 — 설치 건너뜀')

In [ ]:
# [config] 경로와 공용 학습 모듈 로드
import collections
import json
import sys
from pathlib import Path

# 노트북에는 __file__ 이 없다. chatbot.sh 를 마커로 위쪽을 훑어 repo 루트를 찾는다.
ROOT = Path.cwd()
while not (ROOT / 'chatbot.sh').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.train_cls import (  # noqa: E402
    LABEL_NAMES, confusion, ensure_dataset, has_trained_weights,
    load_rows, predict_labels, score, train_classifier,
)

CLS_DIR = ROOT / 'data' / 'cls'
TEST_PATH = ROOT / 'data' / 'test_cls.json'
OUT_PATH = ROOT / 'outputs' / 'cls_output.json'

print('ROOT        =', ROOT)
print('라벨        =', {i: n for i, n in enumerate(LABEL_NAMES)})
print('학습 가중치 =', '있음' if has_trained_weights() else '없음 (아래에서 학습합니다)')

In [ ]:
# [1] 학습 데이터 준비 — 없으면 scripts/build_cls_dataset.py 가 템플릿 x 슬롯으로 생성한다.
#     valid 는 train 과 문자열이 겹치지 않게 분리된다(누수 0 확인).
ensure_dataset()

train_rows = load_rows(CLS_DIR / 'train.json')
valid_rows = load_rows(CLS_DIR / 'valid.json')

print('train %d건 / valid %d건' % (len(train_rows), len(valid_rows)))
print('train 라벨 분포:', sorted(collections.Counter(r['label'] for r in train_rows).items()))
print('train∩valid 겹침:',
      len({r['question'] for r in train_rows} & {r['question'] for r in valid_rows}), '건')
print()
for r in train_rows[:6]:
    print('  [%d %s] %s' % (r['label'], LABEL_NAMES[r['label']], r['question']))

In [ ]:
# [2] 학습 — klue/roberta-base 파인튜닝 후 model/ 에 저장.
#     이미 가중치가 있으면 건너뛴다. 처음부터 다시 학습하려면 FORCE = True 로 바꾼다.
FORCE = False

metrics = train_classifier(force=FORCE)
if metrics is None:
    print('기존 가중치를 사용합니다. 다시 학습하려면 FORCE = True 로 바꾸고 이 셀을 재실행하세요.')

In [ ]:
# [3] 평가 — valid(템플릿 기반)와 eval_natural(사람이 쓴 구어체)을 나눠서 본다.
#     템플릿 데이터로 학습했으므로, 진짜 실력은 eval_natural 쪽 점수가 말해준다.
import torch  # noqa: E402
from transformers import AutoModelForSequenceClassification, AutoTokenizer  # noqa: E402

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(str(ROOT / 'model'))
model = AutoModelForSequenceClassification.from_pretrained(str(ROOT / 'model')).to(device).eval()
print('device =', device)
print()

for name, path in (('valid       ', CLS_DIR / 'valid.json'),
                   ('eval_natural', CLS_DIR / 'eval_natural.json')):
    if not path.exists():
        continue
    rows = load_rows(path)
    gold = [int(r['label']) for r in rows]
    pred = predict_labels(model, tokenizer, [r['question'] for r in rows], device)
    s = score(gold, pred)
    print('[%s] n=%3d  accuracy=%.4f  macro-F1=%.4f'
          % (name, len(rows), s['accuracy'], s['f1_macro']))
    print('               라벨별 F1:',
          {LABEL_NAMES[i]: round(f, 3) for i, f in enumerate(s['f1_per_label'])})
    if 'natural' in name:
        print('               혼동행렬 (행=정답, 열=예측)')
        for i, row in enumerate(confusion(gold, pred)):
            print('                 %-6s %s' % (LABEL_NAMES[i], row))
    print()

In [ ]:
# [4] 제출 산출물 — data/test_cls.json 예측 → outputs/cls_output.json
#     test_cls.json 이 없으면 valid 질문으로 임시 생성해 파이프라인만 확인한다(스모크).
if not TEST_PATH.exists():
    TEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(TEST_PATH, 'w', encoding='utf-8') as f:
        json.dump([{'question': r['question']} for r in load_rows(CLS_DIR / 'valid.json')],
                  f, ensure_ascii=False, indent=2)
    print('test_cls.json 이 없어 valid 로 임시 생성했습니다:', TEST_PATH)

test_rows = load_rows(TEST_PATH)
questions = [r['question'] for r in test_rows]
preds = predict_labels(model, tokenizer, questions, device)

out = [{'id': test_rows[i].get('id', i), 'question': q, 'label': int(p)}
       for i, (q, p) in enumerate(zip(questions, preds))]
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print('%d건 예측 → %s' % (len(out), OUT_PATH))
print('예측 분포:', {LABEL_NAMES[k]: v for k, v in sorted(collections.Counter(preds).items())})
print()
for row in out[:5]:
    print('  ', json.dumps(row, ensure_ascii=False))